In [1]:
!pip install geopandas


Due to MODULEPATH changes, the following have been reloaded:
  1) sqlite/3.44.0     2) udunits/2.2.28

The following have been reloaded with a version change:
  1) JAGS/4.3.2 => JAGS/4.3.0       5) hdf5/1.14.6 => hdf5/1.10.8
  2) R/4.6.0 => R/4.3.0             6) intel/25.3 => intel/21.4
  3) gdal/3.11.4 => gdal/3.6.4      7) netcdf/4.9.3 => netcdf/4.9.2
  4) geos/3.14.1 => geos/3.11.2     8) proj/9.7.0 => proj/9.2.0

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import config as C


def main():
    classes = gpd.read_file(C.TRAINING_GPKG, layer=C.TRAINING_LAYER_CLASSES)
    corrections = gpd.read_file(C.TRAINING_GPKG, layer=C.TRAINING_LAYER_CORRECTIONS)

    training_ids = set(classes["CWNS_ID"].astype(str)) | set(corrections["CWNS_ID"].astype(str))
    print(f"Training universe: {len(training_ids)} unique CWNS_IDs\n")

    loc = pd.read_csv(C.CWNS_DIR / "PHYSICAL_LOCATION.txt", dtype={"CWNS_ID": str}, encoding="latin1")
    loc = loc[loc["CWNS_ID"].isin(training_ids)]
    loc = loc.drop_duplicates(subset="CWNS_ID")

    counts = loc["STATE_CODE"].value_counts().sort_index()
    print(f"{'STATE_CODE':<12}{'plants':>8}")
    for state, n in counts.items():
        print(f"{state:<12}{n:>8}")

    print(f"\nTotal states/territories: {len(counts)}")
    print(f"Total plants (state-known): {counts.sum()}")
    print(f"Largest single state: {counts.idxmax()} with {counts.max()} plants")

    print("\n--- Copy-paste ready ---")
    print("Space-separated (for --array + 01a/01b):")
    print(" ".join(counts.index.tolist()))
    print("\nComma-separated (for 02's --states):")
    print(",".join(counts.index.tolist()))
    print(f"\nArray range for sbatch: --array=0-{len(counts) - 1}")


main()

Training universe: 2616 unique CWNS_IDs

STATE_CODE    plants
AK                15
AL                43
AR                26
AZ                24
CA                84
CO                21
CT                19
DC                 1
DE                 6
FL               163
GA                63
HI                22
IA               117
ID                23
IL                55
IN                83
KS               154
KY                76
LA                27
MA                37
MD                36
ME                26
MI                58
MN                73
MO                79
MS                15
MT                17
NC                90
ND                 6
NE                16
NH                32
NJ                53
NM                13
NV                 3
NY                97
OH               181
OK                61
OR                51
PA                23
PR                22
RI                12
SC                 9
SD                 8
TN               114
TX            